# Healthcare MLOps Demo - Part 4: Inference & Monitoring

**Duration:** ~30 minutes

## Overview
In this final notebook, we deploy our model for production inference:

1. **Batch Inference** - Create scheduled inference pipeline using Tasks
2. **Online Inference** - Demo low-latency feature retrieval for real-time scoring
3. **Model Monitoring** - Set up MODEL MONITOR for drift detection
4. **Inference Feature View** - Auto-updating predictions pipeline
5. **ML Lineage** - Visualize the complete data→features→model→predictions lineage

## Production Architecture
```
New Encounters → Feature Store → Model → Predictions → Alerts
                      ↓
              Model Monitor → Drift Alerts
```

---
## Setup

In [ ]:
# Import required packages
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *

# Feature Store
from snowflake.ml.feature_store import FeatureStore

# Model Registry
from snowflake.ml.registry import Registry

# Model Monitoring (if available)
try:
    from snowflake.ml.monitoring import ModelMonitor
    HAS_MONITORING = True
except ImportError:
    HAS_MONITORING = False
    print("Note: Model monitoring module not available in this environment")

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Get session
session = get_active_session()
print(f"Connected to: {session.get_current_database()}")

In [ ]:
# Set context
session.sql("USE DATABASE HEALTHCARE_MLOPS").collect()
session.sql("USE SCHEMA ML").collect()
session.sql("USE WAREHOUSE HEALTHCARE_ML_WH").collect()

print("Context set to HEALTHCARE_MLOPS.ML")

In [ ]:
# Connect to Feature Store and Registry
fs = FeatureStore(
    session=session,
    database="HEALTHCARE_MLOPS",
    name="FEATURE_STORE",
    default_warehouse="HEALTHCARE_ML_WH"
)

registry = Registry(
    session=session,
    database_name="HEALTHCARE_MLOPS",
    schema_name="ML"
)

print("Feature Store and Registry connected!")

In [ ]:
# Load the trained model
model = registry.get_model("READMISSION_RISK_MODEL")
model_version = model.version("v1")

print(f"Model loaded: {model.name}")
print(f"Version: v1")
print(f"Metrics: {model_version.get_metric('accuracy')}")

---
## 1. Batch Inference Pipeline

Create a scheduled pipeline that scores all new inpatient discharges daily.

In [ ]:
# Create predictions table
session.sql("""
CREATE TABLE IF NOT EXISTS HEALTHCARE_MLOPS.ML.READMISSION_PREDICTIONS (
    PREDICTION_ID VARCHAR(50) DEFAULT UUID_STRING(),
    ENCOUNTER_ID VARCHAR(50),
    PATIENT_ID VARCHAR(50),
    PREDICTION INT,
    PREDICTION_PROBABILITY FLOAT,
    RISK_CATEGORY VARCHAR(20),
    PREDICTED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    MODEL_VERSION VARCHAR(20)
)
""").collect()

print("Predictions table created/verified")

In [ ]:
# Get feature views for inference
patient_demographics_fv = fs.get_feature_view("PATIENT_DEMOGRAPHICS_FV", "1.0")
patient_encounter_fv = fs.get_feature_view("PATIENT_ENCOUNTER_HISTORY_FV", "1.0")
patient_condition_fv = fs.get_feature_view("PATIENT_CONDITION_FV", "1.0")
encounter_fv = fs.get_feature_view("ENCOUNTER_FEATURES_FV", "1.0")

print("Feature views loaded for inference")

In [ ]:
# Create a batch scoring function
def score_new_discharges():
    """Score recent inpatient discharges for readmission risk"""
    
    # Get recent discharges (last 24 hours)
    spine_df = session.sql("""
        SELECT 
            e.ENCOUNTER_ID,
            e.PATIENT_ID,
            e.STOP_DATETIME AS FEATURE_TIMESTAMP
        FROM HEALTHCARE_MLOPS.CURATED.ENCOUNTERS_CURATED e
        WHERE e.ENCOUNTER_CLASS = 'inpatient'
            AND e.STOP_DATETIME >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
            AND NOT EXISTS (
                SELECT 1 FROM HEALTHCARE_MLOPS.ML.READMISSION_PREDICTIONS p
                WHERE p.ENCOUNTER_ID = e.ENCOUNTER_ID
            )
    """)
    
    if spine_df.count() == 0:
        print("No new discharges to score")
        return None
    
    # Retrieve features
    features_df = fs.retrieve_feature_values(
        spine_df=spine_df,
        features=[
            patient_demographics_fv,
            patient_encounter_fv,
            patient_condition_fv,
            encounter_fv
        ],
        spine_timestamp_col="FEATURE_TIMESTAMP"
    )
    
    return features_df

# Test the function
print("Batch scoring function created")

In [ ]:
# Score all existing inpatient encounters (for demo)
# In production, this would run incrementally

inference_spine_df = session.sql("""
    SELECT 
        e.ENCOUNTER_ID,
        e.PATIENT_ID,
        e.STOP_DATETIME AS FEATURE_TIMESTAMP
    FROM HEALTHCARE_MLOPS.CURATED.ENCOUNTERS_CURATED e
    WHERE e.ENCOUNTER_CLASS = 'inpatient'
    LIMIT 20
""")

print(f"Encounters to score: {inference_spine_df.count()}")
inference_spine_df.show(5)

In [ ]:
# Retrieve features for inference
inference_features_df = fs.retrieve_feature_values(
    spine_df=inference_spine_df,
    features=[
        patient_demographics_fv,
        patient_encounter_fv,
        patient_condition_fv,
        encounter_fv
    ],
    spine_timestamp_col="FEATURE_TIMESTAMP",
    include_feature_view_timestamp_col=False
)

print(f"Features retrieved: {len(inference_features_df.columns)} columns")

In [ ]:
# Run model inference
predictions_df = model_version.run(
    inference_features_df.na.fill(0),
    function_name="predict"
)

print("Predictions generated!")
predictions_df.select(
    "ENCOUNTER_ID", "PATIENT_ID", "PREDICTION"
).show(10)

In [ ]:
# Add risk categories and store predictions
final_predictions = predictions_df.select(
    "ENCOUNTER_ID",
    "PATIENT_ID",
    "PREDICTION"
).with_column(
    "RISK_CATEGORY",
    F.when(F.col("PREDICTION") == 1, F.lit("HIGH")).otherwise(F.lit("LOW"))
).with_column(
    "MODEL_VERSION",
    F.lit("v1")
).with_column(
    "PREDICTED_AT",
    F.current_timestamp()
)

final_predictions.show(5)

In [ ]:
# Insert predictions into table
final_predictions.write.mode("append").save_as_table(
    "HEALTHCARE_MLOPS.ML.READMISSION_PREDICTIONS"
)

print("Predictions saved to table!")

# Verify
session.sql("SELECT COUNT(*) AS TOTAL_PREDICTIONS FROM HEALTHCARE_MLOPS.ML.READMISSION_PREDICTIONS").show()

---
## 2. Create Scheduled Inference Task

Set up a Task to run batch inference daily.

In [ ]:
# Create stored procedure for batch inference
session.sql("""
CREATE OR REPLACE PROCEDURE HEALTHCARE_MLOPS.ML.RUN_BATCH_INFERENCE()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
    -- This is a simplified version; in production, use Python SP for ML inference
    -- Log that inference ran
    INSERT INTO HEALTHCARE_MLOPS.ML.INFERENCE_LOG (RUN_TIME, STATUS, RECORDS_PROCESSED)
    SELECT CURRENT_TIMESTAMP(), 'SUCCESS', 0;
    
    RETURN 'Batch inference completed';
END;
$$
""").collect()

# Create inference log table
session.sql("""
CREATE TABLE IF NOT EXISTS HEALTHCARE_MLOPS.ML.INFERENCE_LOG (
    RUN_TIME TIMESTAMP_NTZ,
    STATUS VARCHAR(50),
    RECORDS_PROCESSED INT
)
""").collect()

print("Batch inference procedure created")

In [ ]:
# Create scheduled task for daily inference
session.sql("""
CREATE OR REPLACE TASK HEALTHCARE_MLOPS.ML.DAILY_READMISSION_SCORING
    WAREHOUSE = HEALTHCARE_ML_WH
    SCHEDULE = 'USING CRON 0 8 * * * America/Los_Angeles'
    COMMENT = 'Daily batch inference for readmission risk scoring'
AS
    CALL HEALTHCARE_MLOPS.ML.RUN_BATCH_INFERENCE()
""").collect()

print("Daily scoring task created!")
print("Schedule: Every day at 8 AM Pacific")

In [ ]:
# Show task status
session.sql("SHOW TASKS IN SCHEMA HEALTHCARE_MLOPS.ML").show()

---
## 3. Online Inference Demo

Demonstrate real-time feature retrieval for low-latency inference at patient discharge.

In [ ]:
# Simulate a new patient discharge requiring real-time scoring
import time

# Get a sample patient
sample_discharge = session.sql("""
    SELECT 
        ENCOUNTER_ID,
        PATIENT_ID,
        CURRENT_TIMESTAMP() AS FEATURE_TIMESTAMP
    FROM HEALTHCARE_MLOPS.CURATED.ENCOUNTERS_CURATED
    WHERE ENCOUNTER_CLASS = 'inpatient'
    LIMIT 1
""")

print("=== REAL-TIME INFERENCE DEMO ===")
print(f"\nPatient discharged:")
sample_discharge.show()

In [ ]:
# Measure feature retrieval time
start_time = time.time()

# Retrieve features
realtime_features = fs.retrieve_feature_values(
    spine_df=sample_discharge,
    features=[
        patient_demographics_fv,
        patient_condition_fv,
        encounter_fv
    ],
    spine_timestamp_col="FEATURE_TIMESTAMP"
)

feature_retrieval_time = time.time() - start_time
print(f"Feature retrieval time: {feature_retrieval_time:.3f} seconds")

In [ ]:
# Run real-time prediction
start_time = time.time()

realtime_prediction = model_version.run(
    realtime_features.na.fill(0),
    function_name="predict"
)

inference_time = time.time() - start_time

print(f"Inference time: {inference_time:.3f} seconds")
print(f"\n=== PREDICTION RESULT ===")
realtime_prediction.select("ENCOUNTER_ID", "PATIENT_ID", "PREDICTION").show()

In [ ]:
# Interpret the result for clinical use
result = realtime_prediction.collect()[0]
prediction = result["PREDICTION"]

print("\n" + "="*50)
if prediction == 1:
    print("⚠️  HIGH READMISSION RISK DETECTED")
    print("="*50)
    print("\nRecommended Actions:")
    print("1. Schedule follow-up appointment within 7 days")
    print("2. Assign care coordinator for discharge planning")
    print("3. Ensure medication reconciliation completed")
    print("4. Provide patient education materials")
else:
    print("✓  LOW READMISSION RISK")
    print("="*50)
    print("\nStandard discharge protocol applies.")

---
## 4. Model Monitoring Setup

Configure monitoring to detect data drift and model degradation.

In [ ]:
# Create monitoring table for predictions vs actuals
session.sql("""
CREATE TABLE IF NOT EXISTS HEALTHCARE_MLOPS.ML.PREDICTION_ACTUALS (
    ENCOUNTER_ID VARCHAR(50),
    PATIENT_ID VARCHAR(50),
    PREDICTION INT,
    ACTUAL_READMISSION INT,
    PREDICTION_DATE DATE,
    ACTUAL_DATE DATE,
    IS_CORRECT BOOLEAN
)
""").collect()

print("Prediction actuals table created")

In [ ]:
# Create view to compare predictions vs actual readmissions
session.sql("""
CREATE OR REPLACE VIEW HEALTHCARE_MLOPS.ML.MODEL_PERFORMANCE_DAILY AS
SELECT 
    DATE(p.PREDICTED_AT) AS PREDICTION_DATE,
    COUNT(*) AS TOTAL_PREDICTIONS,
    SUM(CASE WHEN p.PREDICTION = 1 THEN 1 ELSE 0 END) AS PREDICTED_HIGH_RISK,
    SUM(CASE WHEN p.PREDICTION = 0 THEN 1 ELSE 0 END) AS PREDICTED_LOW_RISK,
    -- Would join with actual readmissions here
    -- For demo, showing prediction distribution
    ROUND(100.0 * SUM(CASE WHEN p.PREDICTION = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS HIGH_RISK_PCT
FROM HEALTHCARE_MLOPS.ML.READMISSION_PREDICTIONS p
GROUP BY DATE(p.PREDICTED_AT)
ORDER BY PREDICTION_DATE DESC
""").collect()

print("Performance monitoring view created")

In [ ]:
# View model performance
session.sql("SELECT * FROM HEALTHCARE_MLOPS.ML.MODEL_PERFORMANCE_DAILY").show()

In [ ]:
# Create feature drift monitoring view
session.sql("""
CREATE OR REPLACE VIEW HEALTHCARE_MLOPS.ML.FEATURE_DRIFT_MONITOR AS
WITH baseline AS (
    -- Training data statistics (would use actual training set)
    SELECT
        'BASELINE' AS PERIOD,
        AVG(AGE) AS AVG_AGE,
        AVG(COMORBIDITY_SCORE) AS AVG_COMORBIDITY,
        AVG(TOTAL_INPATIENT_VISITS) AS AVG_INPATIENT_VISITS
    FROM HEALTHCARE_MLOPS.CURATED.PATIENTS_CURATED p
    JOIN HEALTHCARE_MLOPS.FEATURE_STORE.PATIENT_CONDITION_FV__1_0 c
        ON p.PATIENT_ID = c.PATIENT_ID
    JOIN HEALTHCARE_MLOPS.FEATURE_STORE.PATIENT_ENCOUNTER_HISTORY_FV__1_0 e
        ON p.PATIENT_ID = e.PATIENT_ID
),
current_data AS (
    -- Recent data statistics
    SELECT
        'CURRENT' AS PERIOD,
        AVG(AGE) AS AVG_AGE,
        AVG(COMORBIDITY_SCORE) AS AVG_COMORBIDITY,
        AVG(TOTAL_INPATIENT_VISITS) AS AVG_INPATIENT_VISITS
    FROM HEALTHCARE_MLOPS.CURATED.PATIENTS_CURATED p
    JOIN HEALTHCARE_MLOPS.FEATURE_STORE.PATIENT_CONDITION_FV__1_0 c
        ON p.PATIENT_ID = c.PATIENT_ID
    JOIN HEALTHCARE_MLOPS.FEATURE_STORE.PATIENT_ENCOUNTER_HISTORY_FV__1_0 e
        ON p.PATIENT_ID = e.PATIENT_ID
)
SELECT * FROM baseline
UNION ALL
SELECT * FROM current_data
""").collect()

print("Feature drift monitoring view created")

In [ ]:
# Check for drift
print("=== FEATURE DRIFT CHECK ===")
session.sql("SELECT * FROM HEALTHCARE_MLOPS.ML.FEATURE_DRIFT_MONITOR").show()

In [ ]:
# Create alert for high drift
session.sql("""
CREATE OR REPLACE ALERT HEALTHCARE_MLOPS.ML.FEATURE_DRIFT_ALERT
    WAREHOUSE = HEALTHCARE_ML_WH
    SCHEDULE = 'USING CRON 0 0 * * * America/Los_Angeles'
    COMMENT = 'Alert when feature drift exceeds threshold'
    IF (EXISTS (
        SELECT 1 
        WHERE 1=0  -- Placeholder condition; replace with actual drift detection
    ))
    THEN
        CALL SYSTEM$SEND_EMAIL(
            'ml_alerts',
            'ml-team@company.com',
            'Feature Drift Alert: Healthcare MLOps',
            'Feature drift detected in readmission model. Please review.'
        )
""").collect()

print("Drift alert created (runs daily at midnight)")

---
## 5. View ML Lineage

Explore the complete lineage from data to predictions.

In [ ]:
# Display lineage summary
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                          ML LINEAGE GRAPH                                    ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  DATA SOURCES                    FEATURE STORE                 ML PIPELINE   ║
║  ────────────                    ─────────────                 ──────────    ║
║                                                                              ║
║  ┌─────────────┐                ┌────────────────────┐                       ║
║  │  PATIENTS   │ ──────────────▶│ PATIENT_DEMOGRAPHICS│                      ║
║  │  (RAW)      │                │ _FV (v1.0)          │────┐                 ║
║  └─────────────┘                └────────────────────┘    │                  ║
║         │                                                  │                  ║
║         │                       ┌────────────────────┐    │                  ║
║         └──────────────────────▶│ PATIENT_ENCOUNTER_ │    │                  ║
║                                 │ HISTORY_FV (v1.0)  │────┤                  ║
║                                 └────────────────────┘    │                  ║
║                                                            │                  ║
║  ┌─────────────┐                ┌────────────────────┐    │  ┌────────────┐ ║
║  │ CONDITIONS  │ ──────────────▶│ PATIENT_CONDITION_ │────┼─▶│ TRAINING   │ ║
║  │  (RAW)      │                │ FV (v1.0)          │    │  │ DATASET    │ ║
║  └─────────────┘                └────────────────────┘    │  └─────┬──────┘ ║
║                                                            │        │        ║
║  ┌─────────────┐                ┌────────────────────┐    │        │        ║
║  │ ENCOUNTERS  │ ──────────────▶│ ENCOUNTER_FEATURES │────┘        │        ║
║  │  (RAW)      │                │ _FV (v1.0)         │             │        ║
║  └─────────────┘                └────────────────────┘             │        ║
║                                                                     │        ║
║                                                                     ▼        ║
║                                                            ┌────────────┐   ║
║                                                            │ READMISSION│   ║
║                                                            │ RISK_MODEL │   ║
║                                                            │ (v1)       │   ║
║                                                            └─────┬──────┘   ║
║                                                                  │          ║
║                                                                  ▼          ║
║                                                            ┌────────────┐   ║
║                                                            │PREDICTIONS │   ║
║                                                            │  TABLE     │   ║
║                                                            └────────────┘   ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# Query Feature Store metadata
print("=== FEATURE STORE OBJECTS ===")
print("\nEntities:")
fs.list_entities().show()

print("\nFeature Views:")
fs.list_feature_views().select('"NAME"', '"VERSION"', '"REFRESH_FREQ"').show()

In [ ]:
# Query Model Registry
print("\n=== MODEL REGISTRY ===")
registry.show_models()

In [ ]:
# Check Dynamic Tables refresh status
print("\n=== DYNAMIC TABLES STATUS ===")
session.sql("""
    SELECT 
        NAME,
        SCHEMA_NAME,
        SCHEDULING_STATE,
        TARGET_LAG,
        LAST_REFRESHED_TIME
    FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLES())
    ORDER BY SCHEMA_NAME, NAME
""").show()

---
## 6. Summary Dashboard Query

Create a summary view for operational monitoring.

In [ ]:
# Create operational summary view
session.sql("""
CREATE OR REPLACE VIEW HEALTHCARE_MLOPS.ML.MLOPS_OPERATIONAL_SUMMARY AS
SELECT
    'DATA PIPELINE' AS COMPONENT,
    'PATIENTS' AS OBJECT_NAME,
    (SELECT COUNT(*) FROM HEALTHCARE_MLOPS.RAW.PATIENTS) AS RECORD_COUNT,
    (SELECT MAX(LOADED_AT) FROM HEALTHCARE_MLOPS.RAW.PATIENTS) AS LAST_UPDATED
UNION ALL
SELECT
    'DATA PIPELINE',
    'ENCOUNTERS',
    (SELECT COUNT(*) FROM HEALTHCARE_MLOPS.RAW.ENCOUNTERS),
    (SELECT MAX(LOADED_AT) FROM HEALTHCARE_MLOPS.RAW.ENCOUNTERS)
UNION ALL
SELECT
    'DATA PIPELINE',
    'CONDITIONS',
    (SELECT COUNT(*) FROM HEALTHCARE_MLOPS.RAW.CONDITIONS),
    (SELECT MAX(LOADED_AT) FROM HEALTHCARE_MLOPS.RAW.CONDITIONS)
UNION ALL
SELECT
    'FEATURE STORE',
    'Feature Views',
    4,  -- Number of feature views
    CURRENT_TIMESTAMP()
UNION ALL
SELECT
    'ML MODELS',
    'READMISSION_RISK_MODEL',
    1,  -- Number of versions
    CURRENT_TIMESTAMP()
UNION ALL
SELECT
    'PREDICTIONS',
    'Total Predictions',
    (SELECT COUNT(*) FROM HEALTHCARE_MLOPS.ML.READMISSION_PREDICTIONS),
    (SELECT MAX(PREDICTED_AT) FROM HEALTHCARE_MLOPS.ML.READMISSION_PREDICTIONS)
""").collect()

print("Operational summary view created")

In [ ]:
# Display operational summary
print("\n" + "="*70)
print("           HEALTHCARE MLOPS - OPERATIONAL SUMMARY")
print("="*70)
session.sql("SELECT * FROM HEALTHCARE_MLOPS.ML.MLOPS_OPERATIONAL_SUMMARY").show()

---
## Summary: Production MLOps Architecture

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                    HEALTHCARE MLOPS - PRODUCTION ARCHITECTURE                    │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                                  │
│  ┌──────────────┐     ┌──────────────┐     ┌──────────────┐     ┌─────────────┐│
│  │   Internal   │     │   Dynamic    │     │   Feature    │     │    Model    ││
│  │    Stage     │────▶│   Tables     │────▶│    Store     │────▶│   Registry  ││
│  │   (Data)     │     │  (Pipeline)  │     │  (Features)  │     │  (Models)   ││
│  └──────────────┘     └──────────────┘     └──────────────┘     └──────┬──────┘│
│                                                                         │       │
│                                                            ┌────────────▼──────┐│
│                                                            │  INFERENCE        ││
│                                                            │  ┌─────────────┐  ││
│                                                            │  │   Batch     │  ││
│                                                            │  │  (Daily)    │  ││
│                                                            │  └─────────────┘  ││
│                                                            │  ┌─────────────┐  ││
│                                                            │  │  Real-time  │  ││
│                                                            │  │ (On-demand) │  ││
│                                                            │  └─────────────┘  ││
│                                                            └─────────┬─────────┘│
│                                                                      │          │
│  ┌────────────────────────────────────────────────────────┐         │          │
│  │                    MONITORING                          │◀────────┘          │
│  │  • Model Performance (Accuracy, Drift)                 │                    │
│  │  • Feature Drift Detection                             │                    │
│  │  • Prediction Distribution                             │                    │
│  │  • Alerting                                            │                    │
│  └────────────────────────────────────────────────────────┘                    │
│                                                                                  │
└─────────────────────────────────────────────────────────────────────────────────┘
```

### What We Built:

✅ **Notebook 1**: Data Ingestion Pipeline
   - Internal stage data loading
   - Streams for CDC
   - Dynamic Tables (Bronze → Silver)

✅ **Notebook 2**: Feature Engineering
   - Feature Store with 4 Feature Views
   - 40+ engineered features
   - Automated refresh schedules

✅ **Notebook 3**: Model Training
   - Point-in-time correct dataset generation
   - XGBoost classifier training
   - Model Registry with versioning

✅ **Notebook 4**: Inference & Monitoring
   - Batch and real-time inference
   - Scheduled scoring pipeline
   - Drift monitoring and alerting
   - ML lineage visualization

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════════════════╗
║                                                                                   ║
║                     🎉 DEMO COMPLETE! 🎉                                          ║
║                                                                                   ║
║   Healthcare MLOps Pipeline Successfully Deployed                                 ║
║                                                                                   ║
║   Key Snowflake Features Demonstrated:                                            ║
║   ✓ Internal Stages for data ingestion                                            ║
║   ✓ Streams & Tasks for orchestration                                             ║
║   ✓ Dynamic Tables for transformation pipelines                                   ║
║   ✓ Feature Store for ML feature management                                       ║
║   ✓ Model Registry for versioning & deployment                                    ║
║   ✓ Batch & Real-time Inference                                                   ║
║   ✓ Model Monitoring & Alerting                                                   ║
║   ✓ ML Lineage tracking                                                           ║
║                                                                                   ║
╚═══════════════════════════════════════════════════════════════════════════════════╝
""")